In [ ]:
import os
import random
import sys

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn import metrics
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import CLIPModel

/home/tiberiu/Documents/Dizertatie/C2P-CLIP/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset. Local environment detected: using default HF cache.


Downloading: "https://www.now61.com/f/95OefW/C2P_CLIP_release_20240901.zip" to /home/tiberiu/.cache/torch/hub/checkpoints/C2P_CLIP_release_20240901.zip
100%|██████████| 759M/759M [00:31<00:00, 25.0MB/s] 
/home/tiberiu/Documents/Dizertatie/C2P-CLIP/.venv/lib/python3.10/site-packages/torch/hub.py:684: UserWarning: Falling back to the old format < 1.6. This support will be deprecated in favor of default zipfile format introduced in 1.6. Please redo torch.save() to save it in the new zipfile format.
  warnings.warn('Falling back to the old format < 1.6. This support will be '


Real (ID)                , Image number: 1024, mean score: 0.9133
ADM (OOD)                , Image number: 1024, mean score: 0.4376
BigGAN (OOD)             , Image number: 600, mean score: 0.0386
CycleGAN (OOD)           , Image number: 396, mean score: 0.0288
DALLE2 (OOD)             , Image number: 300, mean score: 0.6148
GauGAN (OOD)             , Image number: 1024, mean score: 0.0275
Glide (OOD)              , Image number: 1024, mean score: 0.2592
Midjourney (OOD)         , Image number: 1024, mean score: 0.7412
ProGAN (OOD)             , Image number: 60, mean score: 0.0227
SD14 (OOD)               , Image number: 1024, mean score: 0.3782
SD15 (OOD)               , Image number: 1024, mean score: 0.3604
SDXL (OOD)               , Image number: 600, mean score: 0.6538
StarGAN (OOD)            , Image number: 601, mean score: 0.0270
StyleGAN (OOD)           , Image number: 600, mean score: 0.2341
StyleGAN2 (OOD)          , Image number: 600, mean score: 0.0650


In [ ]:
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount('/content/drive')

    CACHE_DIR = '/content/drive/MyDrive/HuggingFace/cache'
    os.makedirs(CACHE_DIR, exist_ok=True)

    HF_TOKEN = userdata.get('HF_TOKEN')
else:
    from dotenv import load_dotenv

    CACHE_DIR = None

    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')

In [ ]:
seed = 123
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
# C2P-CLIP uses CLIP's own normalization (not ImageNet defaults)
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

# 1. Load the dataset from Hugging Face
if IN_COLAB:
    print(f'Loading dataset. Colab detected: using Drive cache at {CACHE_DIR}')
else:
    print('Loading dataset. Local environment detected: using default HF cache.')

test_data = load_dataset(
    'TheKernel01/AIGC-Detection-Benchmark',
    split='test',
    token=HF_TOKEN,
    cache_dir=CACHE_DIR,
)

generator_mapping = {1: 'SD21', 2: 'SDXL', 3: 'SD3', 4: 'DALLE3', 5: 'Midjourney 6'}

In [ ]:
def sim_auc(similarities, datasets):
    """
    Calculate AUC and FPR95 for multiple OOD datasets against ID dataset.

    Args:
        similarities (list): List of similarity arrays, first one is ID dataset
        datasets (list): List of dataset names

    Returns:
        tuple: (average_auc, average_fpr95)
    """
    if len(similarities) != len(datasets):
        raise ValueError(
            'Number of similarities arrays must match number of dataset names'
        )

    if len(similarities) < 2:
        raise ValueError('At least 2 datasets (ID and OOD) are required')

    similarities = np.array(similarities, dtype=object)
    id_confi = similarities[0]

    auc_scores = []
    fpr_scores = []

    for ood_confi, dataset in zip(similarities[1:], datasets[1:]):
        auroc, fpr_95 = calculate_auc_metrics(id_confi, ood_confi)
        auc_scores.append(auroc)
        fpr_scores.append(fpr_95)
        print(f'Dataset: {dataset:<25} | AUC: {auroc:.4f} | FPR95: {fpr_95:.4f}')

    avg_auc = np.mean(auc_scores)
    avg_fpr = np.mean(fpr_scores)

    print('-' * 60)
    print(f'Average AUC: {avg_auc:.4f} | Average FPR95: {avg_fpr:.4f}')

    return avg_auc, avg_fpr


def sim_ap(similarities, datasets):
    """
    Calculate Average Precision for multiple OOD datasets against ID dataset.

    Args:
        similarities (list): List of similarity arrays, first one is ID dataset
        datasets (list): List of dataset names

    Returns:
        float: average AP score
    """
    if len(similarities) != len(datasets):
        raise ValueError(
            'Number of similarities arrays must match number of dataset names'
        )

    if len(similarities) < 2:
        raise ValueError('At least 2 datasets (ID and OOD) are required')

    similarities = np.array(similarities, dtype=object)
    id_confi = similarities[0]

    ap_scores = []

    for ood_confi, dataset in zip(similarities[1:], datasets[1:]):
        aver_p = calculate_average_precision(id_confi, ood_confi)
        ap_scores.append(aver_p)
        print(f'Dataset: {dataset:<25} | AP: {aver_p:.4f}')

    avg_ap = np.mean(ap_scores)
    print('-' * 40)
    print(f'Average AP: {avg_ap:.4f}')

    return avg_ap


def calculate_auc_metrics(id_conf, ood_conf):
    """
    Calculate AUROC and FPR at 95% TPR for binary classification.

    Args:
        id_conf (np.ndarray): Confidence scores for ID (in-distribution) samples
        ood_conf (np.ndarray): Confidence scores for OOD (out-of-distribution) samples

    Returns:
        tuple: (auroc, fpr_at_95_tpr)
    """
    all_conf = np.concatenate([id_conf, ood_conf])
    # ID samples are positive (1), OOD samples are negative (0)
    labels = np.concatenate([np.ones(len(id_conf)), np.zeros(len(ood_conf))])

    fpr, tpr, _ = metrics.roc_curve(labels, all_conf)
    auroc = metrics.auc(fpr, tpr)

    tpr_threshold = 0.95
    valid_indices = tpr >= tpr_threshold
    if np.any(valid_indices):
        fpr_at_95 = fpr[np.argmax(valid_indices)]
    else:
        fpr_at_95 = fpr[-1]
        print(f'Warning: 95% TPR not achievable. Max TPR: {tpr[-1]:.3f}')

    return auroc, fpr_at_95


def calculate_average_precision(id_predictions, ood_predictions):
    all_predictions = np.concatenate([id_predictions, ood_predictions])
    # ID samples are positive (1), OOD samples are negative (0)
    labels = np.concatenate(
        [np.ones(len(id_predictions)), np.zeros(len(ood_predictions))]
    )
    average_precision = metrics.average_precision_score(labels, all_predictions)
    return average_precision

In [ ]:
# 2. Create a custom PyTorch Dataset wrapper for the Hugging Face dataset
class HFImageDataset(Dataset):
    def __init__(self, hf_data, transform=None):
        self.hf_data = hf_data
        self.transform = transform

    def __len__(self):
        return len(self.hf_data)

    def __getitem__(self, idx):
        item = self.hf_data[idx]

        # Ensure the image is in 3-channel RGB format
        image = item['image'].convert('RGB')
        label = item['label']

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
class C2P_CLIP(nn.Module):
    def __init__(self, name='openai/clip-vit-large-patch14', num_classes=1):
        super(C2P_CLIP, self).__init__()
        self.model = CLIPModel.from_pretrained(name)
        del self.model.text_model
        del self.model.text_projection
        del self.model.logit_scale

        self.model.vision_model.requires_grad_(False)
        self.model.visual_projection.requires_grad_(False)
        self.model.fc = nn.Linear(768, num_classes)
        torch.nn.init.normal_(self.model.fc.weight.data, 0.0, 0.02)

    def encode_image(self, img):
        vision_outputs = self.model.vision_model(
            pixel_values=img,
            output_attentions=self.model.config.output_attentions,
            output_hidden_states=self.model.config.output_hidden_states,
            return_dict=self.model.config.use_return_dict,
        )
        pooled_output = vision_outputs[1]  # pooled_output
        image_features = self.model.visual_projection(pooled_output)
        return image_features

    def forward(self, img):
        image_embeds = self.encode_image(img)
        image_embeds = image_embeds / image_embeds.norm(p=2, dim=-1, keepdim=True)
        return self.model.fc(image_embeds)


class C2P_CLIP_Detector:
    def __init__(self, clip_name='openai/clip-vit-large-patch14'):
        state_dict = torch.hub.load_state_dict_from_url(
            'https://www.now61.com/f/95OefW/C2P_CLIP_release_20240901.zip',
            map_location='cpu',
            progress=True,
        )
        # state_dict = torch.load(
        #     './models/C2P_CLIP-GenImage_release_20250224.pth',
        #     map_location='cpu',
        #     weights_only=True,
        # )
        self.model = C2P_CLIP(name=clip_name, num_classes=1)
        self.model.load_state_dict(state_dict, strict=True)
        self.model.cuda()
        self.model.eval()

    @torch.no_grad()
    def detect(self, data):
        # Returns sigmoid probability: high = fake, low = real
        return self.model(data).sigmoid().flatten()

In [ ]:
transform_C2P = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ]
)

In [ ]:
# 3. Filter the dataset into Real (ID) and specific AI-generated (OOD) subsets
all_generators = np.array(test_data['generator'])

# 3a. Extract Real (ID) dataset
real_indices = np.nonzero(all_generators == 0)[0]
real_hf = test_data.select(real_indices)
real_dataset = HFImageDataset(real_hf, transform=transform_C2P)

evaluation_datasets = [('Real (ID)', real_dataset)]

# 3b. Extract Fake (OOD) datasets
for gen_id, gen_name in generator_mapping.items():
    fake_indices = np.nonzero(all_generators == gen_id)[0]
    fake_hf = test_data.select(fake_indices)
    fake_dataset = HFImageDataset(fake_hf, transform=transform_C2P)
    evaluation_datasets.append((f'{gen_name} (OOD)', fake_dataset))

test_datasets = [name for name, _ in evaluation_datasets]
batch_size = 256 if IN_COLAB else 32

c2p_detector = C2P_CLIP_Detector()

In [ ]:
# NOTE on score convention:
# C2P-CLIP outputs a sigmoid probability where high = fake (AI-generated).
# To align with the AUC/AP helpers (which treat ID=real as positive=1),
# we invert the score: score = 1 - p(fake), so real images score high.
with torch.no_grad():
    sim_datasets = []

    for dataset_name, dataset_obj in evaluation_datasets:
        data_loader = DataLoader(
            dataset_obj,
            batch_size=batch_size,
            shuffle=False,
            num_workers=8,
            pin_memory=True,
            persistent_workers=True,
        )

        scores = []
        total_num = 0

        for i, (samples, _) in enumerate(data_loader):
            samples = samples.cuda()
            total_num += len(samples)

            # Invert: high score = real (ID-positive convention)
            score = 1.0 - c2p_detector.detect(samples)
            scores.append(score)

            if total_num >= 1000:
                break

        if len(scores) > 0:
            scores = torch.cat(scores, dim=0)
            print(
                f'{dataset_name:<25}, Image number: {scores.shape[0]}, mean score: {scores.mean().item():.4f}'
            )
            sim_datasets.append(scores.cpu().numpy())
        else:
            print(f'Warning: {dataset_name} is empty. Check your label filtering!')

    print('\n' + '=' * 60)
    print('Detection Results AUC (Per Generator):')
    print('=' * 60)
    sim_auc(sim_datasets, test_datasets)

    print('\n' + '=' * 60)
    print('Detection Results AP (Per Generator):')
    print('=' * 60)
    sim_ap(sim_datasets, test_datasets)